In [1]:
import sys
from pathlib import Path

sys.path.append("..")
from scripts.semantic_searcher import SemanticSearcher
import json
import pandas as pd

In [2]:
searcher = SemanticSearcher()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 10000 listings...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [3]:
ROOT = Path("..").resolve()
QUERIES_PATH = ROOT / "data" / "processed" / "user_queries_50.json"
OUT_DIR = ROOT / "data" / "analyze_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "semantic_search_emb_vs_bm25_top5.json"

TOP_K = 5

with QUERIES_PATH.open(encoding="utf-8") as f:
    user_data = json.load(f)

queries = user_data["queries"]


def _pack_hits(results, ids_scores):
    """Flatten (remark, score) + (L_ListingID, score) into JSON-serializable rows."""
    rows = []
    for (remark, score), (listing_id, _) in zip(results, ids_scores):
        lid = listing_id
        if pd.notna(lid) and isinstance(lid, float) and lid == int(lid):
            lid = int(lid)
        rows.append(
            {
                "listing_id": str(lid),
                "score": float(score),
                "remark": remark if isinstance(remark, str) else str(remark),
            }
        )
    return rows


rows_out = []
for i, item in enumerate(queries):
    q = item["query"]
    res_emb, ids_emb, lat_emb = searcher.search_emb(q, top_k=TOP_K)
    res_bm, ids_bm, lat_bm = searcher.search_bm25(q, top_k=TOP_K)
    rows_out.append(
        {
            "index": i,
            "journey": item.get("journey"),
            "intent": item.get("intent"),
            "query": q,
            "embedding": {
                "latency_ms": float(lat_emb),
                "top_k": _pack_hits(res_emb, ids_emb),
            },
            "bm25": {
                "latency_ms": float(lat_bm),
                "top_k": _pack_hits(res_bm, ids_bm),
            },
        }
    )

payload = {
    "meta": {
        "queries_source": str(QUERIES_PATH.as_posix()),
        "n_queries": len(rows_out),
        "top_k": TOP_K,
        "listings_rows": int(searcher.listings.shape[0]),
    },
    "results": rows_out,
}

with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

OUT_PATH

WindowsPath('D:/nlp-internship/data/analyze_result/semantic_search_emb_vs_bm25_top5.json')